# 0911 10일차

## 1. 스케일링의 목적 - 범위 맞추기

> 값의 **범위**를 맞춰서, 어떤 컬럼도 크기만으로 우위를 갖지 못하게 하는 것

9일차 §4의 캘리포니아 데이터가 그대로 근거가 됨

```
MedInc        0.500 ~       15.000      <- 소득
Population    3.000 ~    35682.000      <- 인구
```

→ 모델이 `w`를 갱신할 때 **숫자가 큰 컬럼이 loss와 기울기를 지배**함. 인구는 조금만 움직여도 loss가 크게 변하고, 소득은 아무리 움직여봐야 묻힘

→ 범위를 맞추면 경사 하강법이 **더 빨리 수렴함.** wine 학습 시간으로 실측됨 (119.74초 → 25.08초)

### 이상치 - MinMax가 오히려 약함

"스케일링을 안 하면 이상치 영향을 많이 받는다"고 정리했는데 **반대임**

MinMaxScaler는 MIN/MAX를 **직접** 쓰므로 이상치에 가장 약한 스케일러

9일차 §5에서 봤던 그 `AveOccup` 컬럼으로 확인해보면

```
   원본                      MinMax 적용 후
   25%      2.430      →       0.001398
   50%      2.818      →       0.001711
   75%      3.282      →       0.002084
   99%      5.395      →       0.003784
  100%   1243.333      →       1.000000      <- 이상치 1개
```

| | |
|---|---|
| 0 ~ 0.01 구간에 몰린 비율 | **99.89%** |
| 0.5 ~ 1.0 구간의 개수 | **1개** / 20,640 |

→ 이상치 하나가 1.0을 차지하고 **나머지 20,639개가 0.002 근처에 몰림**

→ 이상치는 스케일링으로 해결되지 않음. 4일차 §6의 이상치 처리가 **먼저**, 스케일링이 **나중**

### 범위 vs 분포 - 모양은 그대로

"각 컬럼의 분포를 동일하게 맞춘다"도 정확하지 않음. MinMaxScaler는 **선형변환**이라 분포 모양을 바꾸지 못함

```
왜도(skew)   원본 97.632465  →  스케일 후 97.632465     (완전히 동일)
원본과의 상관계수 : 1.0000000000
```

→ 축의 눈금만 바꿔 단 것. **히스토그램 모양은 그대로**

실제로 MinMax를 적용한 뒤에도 컬럼별 분포는 전혀 맞지 않음

```
컬럼            중앙값      IQR폭
MedInc         0.2093     0.1503
HouseAge       0.5490     0.3725
AveOccup       0.0017     0.0007      <- 여전히 제각각
```

→ MinMaxScaler가 맞춰주는 건 **양 끝(0과 1)뿐**

| | 맞춰지는 것 | 안 맞춰지는 것 |
|---|---|---|
| MinMaxScaler | 최소값·최대값 | 중앙값, 퍼진 정도, 분포 모양 |

→ 그래서 표현은 "분포를 맞춘다"가 아니라 **"범위를 맞춘다"**가 맞음

In [ ]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import skew

d = fetch_california_housing()
x = d.data
col = x[:, d.feature_names.index('AveOccup')]

s = MinMaxScaler().fit_transform(col.reshape(-1, 1)).ravel()

# 이상치 하나 때문에 나머지가 전부 눌림
print(np.percentile(col, [25, 50, 75, 100]))    # [   2.429   2.818   3.282 1243.333]
print(np.percentile(s,   [25, 50, 75, 100]))    # [0.0014 0.0017 0.0021 1.0    ]
print((s < 0.01).mean())                        # 0.998886...  <- 99.89%가 0~0.01에 몰림

# 분포 모양은 바뀌지 않음 (선형변환)
print(skew(col), skew(s))                       # 97.632465 97.632465  <- 같음
print(np.corrcoef(col, s)[0, 1])                # 1.0

## 2. StandardScaler - 평균 0, 표준편차 1

$$x' = \frac{x - \text{평균}}{\text{표준편차}}$$

```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(x_train)                     # 평균, 표준편차를 계산
x_train = scaler.transform(x_train)
x_test = scaler.transform(x_test)       # 9일차 §5와 순서는 동일
```

→ 분자 `x - 평균`이 **중심을 0으로 옮기고**, 분모 `표준편차`가 **퍼진 정도를 1로 맞춤**

→ 평균만 0으로 옮기면 위치만 이동할 뿐임. 인구(35,682)는 여전히 소득(15)보다 큼. **표준편차로 나누는 쪽이 실제로 스케일을 맞추는 일을 함**

```
StandardScaler 후
  컬럼             평균     표준편차
  MedInc       0.000000   1.000000
  Population  -0.000000   1.000000
  AveOccup     0.000000   1.000000      <- 전부 정확히 0과 1
```

### MinMax와 보완 관계

| | 맞춰지는 것 | 안 맞춰지는 것 |
|---|---|---|
| **MinMaxScaler** | 최소·최대 (0, 1) | 평균, 표준편차 |
| **StandardScaler** | 평균, 표준편차 (0, 1) | 최소, 최대 |

StandardScaler를 하면 범위는 컬럼마다 제각각이 됨

```
MedInc      -1.774 ~    5.858
HouseAge    -2.196 ~    1.856
AveOccup    -0.229 ~  119.419      <- 이상치가 그대로 남아 있음
```

→ 0~1로 딱 떨어지지 않는 게 정상. **범위를 고정하는 스케일러가 아님**

### 이상치 - StandardScaler가 강함

§1에서 MinMaxScaler는 이상치 하나가 단위(MAX)를 정해버린다고 했음

그 결과 8개 컬럼의 **실효 표준편차**가 이렇게 벌어짐

| | MinMax 후 표준편차 | Standard 후 표준편차 |
|---|---|---|
| HouseAge | 0.2468 | 1.0 |
| MedInc | 0.1310 | 1.0 |
| AveRooms | 0.0175 | 1.0 |
| **AveOccup** | **0.0084** | 1.0 |

→ MinMax 쪽은 가장 큰 컬럼과 가장 작은 컬럼이 **29배** 차이. 범위는 맞췄는데 **평소 변동폭은 안 맞음**

→ StandardScaler는 표준편차가 단위라 **8개 컬럼 모두 정확히 1**. 컬럼 간 평소 변동폭이 같아짐

→ **이상치가 있으면 MinMax보다 StandardScaler 쪽**

### cf) 정규분포 - 되는 게 아님

이름이 "표준화"라 오해하기 쉬운데, 이것도 **선형변환**이라 분포 모양을 바꾸지 못함

```
왜도(skew)      원본      Standard     MinMax
AveOccup     97.6325      97.6325    97.6325      <- 셋 다 동일
AveRooms     20.6964      20.6964    20.6964
```

정규분포라면 ±1 안에 68%가 들어와야 하는데

```
AveOccup    ±1 안  99.9%
Longitude   ±1 안  58.8%
```

→ 평균 0·표준편차 1이 된 것뿐, **종 모양이 된 게 아님**

→ §1의 "선형변환이라 분포 모양은 그대로"가 StandardScaler에도 똑같이 적용됨

In [ ]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from scipy.stats import skew

d = fetch_california_housing()
x = d.data
j = d.feature_names.index('AveOccup')

z = StandardScaler().fit_transform(x)
m = MinMaxScaler().fit_transform(x)

# Standard - 평균 0, 표준편차 1
print(z[:, j].mean(), z[:, j].std())    # 0.0 1.0
print(z[:, j].min(), z[:, j].max())     # -0.229 119.419   <- 범위는 안 맞춰짐

# MinMax - 범위는 맞지만 평소 변동폭이 컬럼마다 다름
print(m[:, j].min(), m[:, j].max())     # 0.0 1.0
print(m[:, j].std())                    # 0.0084           <- 거의 안 씀
print(m[:, 1].std())                    # 0.2468  HouseAge <- 29배 차이

# 둘 다 분포 모양은 그대로
print(skew(x[:, j]), skew(z[:, j]), skew(m[:, j]))  # 97.6325 세 번

## 3. MaxAbsScaler - 최대 절댓값으로 나누기

$$x' = \frac{x}{\max(|x|)}$$

```python
from sklearn.preprocessing import MaxAbsScaler
```

→ 9일차 §4에서 "가장 큰 수로 나눠주기"라고 적었던 **그 방식 그대로**. MinMax는 거기서 최소값이 0이 안 나오는 걸 `- MIN`으로 보정한 것이고, MaxAbs는 **보정 없이 나누기만** 함

→ 범위는 **-1 ~ 1**. 빼기가 없으니 원점 기준으로 줄이기만 함

| | MaxAbs만 갖는 성질 |
|---|---|
| **0이 0으로 유지** | 대부분이 0인 희소 데이터(텍스트 TF-IDF 등)에서 희소성이 안 깨짐 |
| **부호 유지** | 음수 컬럼은 음수인 채로 |

→ MinMax를 희소 데이터에 쓰면 0이 0 아닌 값으로 바뀜. **MaxAbs의 주 용도가 이것**

### 0에서 먼 컬럼 - 범위를 거의 못 씀

캘리포니아에 둘 다 적용해보면

```
컬럼          MaxAbs 범위          MinMax 범위    MaxAbs가 쓰는 폭
MedInc      0.0333 ~  1.0000      0 ~ 1            0.9667
Population  0.0001 ~  1.0000      0 ~ 1            0.9999
Latitude    0.7757 ~  1.0000      0 ~ 1            0.2243   <-
Longitude  -1.0000 ~ -0.9193      0 ~ 1            0.0807   <-
```

→ 앞 6개는 최소값이 0에 가까워 **MinMax와 사실상 같음**

→ 그런데 경도는 -124.35 ~ -114.31이라 124.35로 나누면 전부 -1 근처에 뭉침. -1~1 중 **8%만 씀**

→ 컬럼이 **0에서 멀리 떨어져 있으면** MaxAbs는 범위를 거의 쓰지 못함. MinMax가 `- MIN`을 하는 이유가 이것

### cf) 이상치 - MinMax와 똑같이 약함

§1의 `AveOccup`을 두 스케일러로 비교하면

```
           중앙값       최대
MinMax    0.001711     1.0
MaxAbs    0.002267     1.0      <- 거의 동일
```

→ **둘 다 최대값을 직접 쓰기 때문.** 이상치 대응은 여전히 StandardScaler 쪽(§2)

→ 성능은 데이터에 따라 다름. 캘리포니아처럼 대부분 양수면 두 스케일러 차이가 컬럼별 이동·배율뿐이라 **결과가 비슷하게 나오는 게 보통**

In [ ]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import MinMaxScaler, MaxAbsScaler

d = fetch_california_housing()
x = d.data

m = MinMaxScaler().fit_transform(x)
a = MaxAbsScaler().fit_transform(x)

j = d.feature_names.index('Longitude')
print(a[:, j].min(), a[:, j].max())      # -1.0 -0.9193   <- 8% 폭만 씀
print(m[:, j].min(), m[:, j].max())      # 0.0 1.0

# 0 은 0 그대로 (희소 데이터에서 중요)
print(MaxAbsScaler().fit(np.array([[0.], [10.]])).transform(np.array([[0.]])))  # [[0.]]

## 4. RobustScaler - 중앙값과 IQR로 나누기

특정 값이 지나치게 높으면 평균이 자료 전체를 대표하지 못함 → 중앙값을 씀 (`AveBedrms`는 데이터의 **74%가 평균보다 작음**)

$$x' = \frac{x - \text{중앙값}}{\text{IQR}}$$

```python
from sklearn.preprocessing import RobustScaler
```

→ IQR = 75% 지점 - 25% 지점. **가운데 50%가 퍼진 폭**

→ 기준값이 중앙값과 IQR이라 **극단값이 아무리 커도 흔들리지 않음.** 이상치에 강한 건 여기서 이미 정해짐

```
RobustScaler 가 쓰는 기준 : 중앙값 2.8181 / IQR 0.8525
MinMax 가 쓰는 기준       : MIN 0.6923 / MAX 1243.3333    <- 이상치가 기준에 직접 들어감
```

→ §1~§3의 세 스케일러는 전부 MIN/MAX나 표준편차를 썼음. **이상치가 기준값 계산에 그대로 들어가는 구조**

### 효과 - 가운데 50%가 폭 1을 차지

§1의 `AveOccup`(중앙값 2.818, IQR 0.853, 최대 1243.3)을 네 스케일러로 비교

| | 가운데 50%가 차지하는 폭 | 이상치가 가는 위치 |
|---|---|---|
| MinMax | 0.000686 | 1.00 |
| MaxAbs | 0.000686 | 1.00 |
| Standard | 0.082085 | 119.42 |
| **Robust** | **1.000000** | 1455.12 |

→ **Robust도 이상치를 없애지 않음.** 1243은 1455로 바뀔 뿐 여전히 멀리 떨어져 있음

→ 대신 단위를 중앙값·IQR로 잡으니 **가운데 50%가 폭 1을 차지함.** §1에서 MinMax가 99.89%를 0.002 근처에 몰아넣던 문제가 해결됨

→ IQR 폭이 정확히 1.0인 건 **IQR로 나눴기 때문**

### 주의) 결과가 좋다 ≠ 스케일러가 좋다

"RobustScaler를 썼는데 결과가 잘 나오면 이상치에 강하다는 뜻"은 **방향이 반대**

→ 이상치에 강한 건 위에서 봤듯 **설계상 이미 정해진 성질(전제)**이지, 결과로 알아내는 게 아님

→ 결과가 좋아졌을 때 알게 되는 건 스케일러가 아니라 **내 데이터**

> RobustScaler로 바꿨더니 좋아졌다 → **그 데이터에 이상치가 있었다**는 뜻

→ 반대 방향도 성립하지 않음. Robust로 안 좋아졌다고 이상치가 없는 건 아님. **이상치가 있어도 그게 예측에 중요한 정보면** 눌러버리는 게 손해일 수 있음

In [ ]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

d = fetch_california_housing()
col = d.data[:, d.feature_names.index('AveOccup')].reshape(-1, 1)

for nm, S in [("MinMax", MinMaxScaler()), ("Standard", StandardScaler()), ("Robust", RobustScaler())]:
    c = S.fit_transform(col).ravel()
    iqr = np.percentile(c, 75) - np.percentile(c, 25)
    print(nm, round(iqr, 6), round(c.max(), 2))
# MinMax   0.000686     1.0
# Standard 0.082085   119.42
# Robust   1.0       1455.12    <- 가운데는 제 폭, 이상치는 그대로

r = RobustScaler().fit(col)
print(r.center_, r.scale_)      # [2.8181] [0.8525]   <- 중앙값, IQR

## 5. 모델 저장과 불러오기 - `save` / `load_model`

훈련에 300초씩 걸리는 모델을 매번 다시 돌릴 수는 없음 (9일차 §1의 산탄데르)

```python
from tensorflow.keras.models import Sequential, load_model

model.save(path + 'keras29_1_save_model.keras')     # 저장
model = load_model(path + 'keras29_1_save_model.keras')   # 불러오기
```

→ 확장자 `.keras`가 현재 표준 포맷. 파일 **하나**에 구조·가중치·compile 정보가 함께 들어감

→ `load_model`은 `Sequential()`부터 `add`까지를 **대신함.** 불러온 뒤에는 모델 구성 코드가 필요 없음

`keras29_1` ~ `keras29_4`가 이걸 두 경우로 나눠 실습한 것

| 파일 | 하는 일 | `save` 위치 |
|---|---|---|
| `29_1` | 모델 구성 후 저장 | **`fit` 전** |
| `29_2` | 불러와서 **훈련** → 평가 | |
| `29_3` | 구성 → 훈련 → 저장 | **`fit` 후** |
| `29_4` | 불러와서 **바로 평가** | |

→ 경로는 `'C:\study\_save\keras29/'`. 5일차 §0의 이스케이프 문제라 `/`로 통일하는 게 안전함

### 저장 시점 - 훈련 전이냐 후냐

두 파일을 실제로 열어 첫 층 가중치를 찍어보면

```
29_1 (fit 전 저장)   첫 층 가중치 [-0.08221 -0.50429  0.19401]    <- 무작위 초기값
29_3 (fit 후 저장)   첫 층 가중치 [-2.5696   1.24182 -1.10475]    <- 학습된 값
```

→ **"훈련 전에는 구조만 저장된다"는 오해.** 가중치도 저장됨. 다만 그게 `fit` 전이라 **무작위 초기값**일 뿐

→ 그래서 `29_2`는 불러온 뒤 `fit`을 해야 쓸모가 있고, `29_4`는 `fit` 없이 바로 `evaluate`가 됨

대신 빠지는 건 **compile 정보**임

```
29_1  model.loss -> None      <- compile 전에 저장했으니
29_3  model.loss -> 'mse'
```

`29_1`을 불러와 `compile` 없이 평가하면

```
ValueError: You must call `compile()` before using the model.
```

→ `29_4`가 `compile` 한 줄 없이 바로 `evaluate`할 수 있는 이유. **compile까지 파일에 들어 있음**

### cf) 불러온 모델은 똑같은 결과를 냄

`29_3`이 저장한 모델을 두 번 불러와 평가하면

```
1회차 evaluate : 0.5612404942513
2회차 evaluate : 0.5612404942513
```

→ 가중치가 파일에 그대로 들어 있으니 **같은 값**이 나옴

→ 그러니 `29_3`과 `29_4`의 결과가 다르다면 **주석부터 확인할 것.** 실제로 `29_3`의 주석(`loss : 0.5186...`)은 `29_2`에서 복사한 값이고, 그 파일의 진짜 실행 결과는 `29_4`에 적힌 `0.5612...`임

In [ ]:
import numpy as np
from tensorflow.keras.models import load_model

p = 'C:/study/_save/keras29/'

m1 = load_model(p + 'keras29_1_save_model.keras')   # fit 전에 저장한 것
m3 = load_model(p + 'keras29_3_save_model.keras')   # fit 후에 저장한 것

print(m1.layers[0].get_weights()[0].ravel()[:3])    # [-0.08221 -0.50429  0.19401]  무작위
print(m3.layers[0].get_weights()[0].ravel()[:3])    # [-2.5696   1.24182 -1.10475]  학습됨

print(m1.loss)      # None    <- compile 전에 저장 -> 불러와도 compile 해야 함
print(m3.loss)      # mse     <- compile 정보까지 들어 있음

# m1.evaluate(x_test, y_test)
# ValueError: You must call `compile()` before using the model.